<a href="https://colab.research.google.com/github/will-mccormack/CS-M148-Proj/blob/main/COM_SCI_M148_NN_with_genre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, utils
from sklearn.preprocessing import StandardScaler
import time

import warnings
warnings.filterwarnings("ignore")

In [2]:
train_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/train.csv"
validation_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/validation.csv"

#train_data = pd.read_csv(train_filepath)
#validation_data = pd.read_csv(train_filepath)

In [3]:
# device setup to find gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
def load_data(train_path, val_path):
  train_data = pd.read_csv(train_path)
  val_data = pd.read_csv(val_path)
  train_data['explicit'] = train_data['explicit'].astype(int)
  val_data['explicit'] = val_data['explicit'].astype(int)
  train_data = pd.get_dummies(train_data, columns=['track_genre'], dtype=float)
  val_data = pd.get_dummies(val_data, columns=['track_genre'], dtype=float)
  train_cols =train_data.columns
  val_data = val_data.reindex(columns=train_cols, fill_value=0)
  return train_data, val_data

In [19]:
class SpotifyDataset(Dataset):
  def __init__(self, dataframe,scaler=None,is_train=True):
    # split x and y
    self.x = dataframe.drop(columns=['popularity']).values
    self.y = dataframe['popularity'].values
    if is_train:
      self.scaler = StandardScaler()
      self.x = self.scaler.fit_transform(self.x) #scale data
    else:
      self.scaler = scaler
      self.x = self.scaler.transform(self.x)

  def __len__(self):
    return len(self.x)

  def __getitem__(self, idx):
    features = self.x[idx]
    label = self.y[idx]
    features_tensor = torch.tensor(features, dtype=torch.float32)
    label_tensor = torch.tensor(label, dtype=torch.float32)
    return features_tensor, label_tensor

In [20]:
train_data, val_data = load_data(train_filepath, validation_filepath)
train_dataset = SpotifyDataset(train_data, is_train=True)
val_dataset = SpotifyDataset(val_data, scaler=train_dataset.scaler, is_train=False)

In [21]:
BATCH_SIZE = 1024
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,pin_memory=True)

input_size = train_dataset.x.shape[1]
print(f"Input size: {input_size}")

Input size: 127


# NN

In [16]:
model = nn.Sequential(
    nn.Linear(input_size,128),
    nn.ReLU(),
    nn.Linear(128,64),
    nn.ReLU(),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,1)
)

model = model.to(device) # move to GPU

loss_type = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 100
patience = 10
best_val_loss = float('inf')
epochs_no_improve = 0

print("Starting Training")

start_time = time.time()

for epoch in range(num_epochs):
  model.train()
  epoch_loss = 0.0

  for features, labels in train_loader:
    features, labels = features.to(device), labels.to(device) # move data to GPU

    optimizer.zero_grad()
    # forward pass, predicted
    predictions = model(features)
    # loss
    loss = loss_type(predictions, labels.view(-1,1))
    #backprop, gradients and update weights

    loss.backward()
    optimizer.step()
    #update loss
    epoch_loss += loss.item()

  # validation test
  model.eval()
  val_loss = 0.0
  with torch.no_grad():
    for features, labels in val_loader:
      features, labels = features.to(device), labels.to(device)
      predictions = model(features)
      val_loss += loss_type(predictions, labels.view(-1,1)).item()

  avg_train_loss = epoch_loss / len(train_loader)
  avg_val_loss = val_loss / len(val_loader)

  if (epoch + 1) % 5 == 0:
    print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")
  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    patience_counter = 0
  else:
    patience_counter += 1
    if patience_counter >= patience:
      print(f"Early stopping at epoch {epoch+1}")
      break

print("Training finished")
print(f"Done in {time.time() - start_time:.2f}s")

Starting Training
Epoch 5/100 | Train: 365.3349 | Val: 374.2453
Epoch 10/100 | Train: 359.3981 | Val: 370.0954
Epoch 15/100 | Train: 353.8896 | Val: 366.8715
Epoch 20/100 | Train: 349.4824 | Val: 364.5378
Epoch 25/100 | Train: 344.6942 | Val: 362.0238
Epoch 30/100 | Train: 338.3211 | Val: 360.3964
Epoch 35/100 | Train: 330.8559 | Val: 357.9669
Epoch 40/100 | Train: 322.8853 | Val: 359.3142
Epoch 45/100 | Train: 316.9747 | Val: 359.6014
Epoch 50/100 | Train: 310.0704 | Val: 358.1813
Epoch 55/100 | Train: 302.9556 | Val: 357.6593
Epoch 60/100 | Train: 298.0741 | Val: 361.7249
Early stopping at epoch 61
Training finished
Done in 143.52s
